# Stores (Part 1) — Long-Term Cross-Conversation Memory

This notebook introduces **Stores** — a mechanism for keeping information about users **across multiple conversation threads**. This is different from checkpointers, which only remember things within the same thread.

## Key concepts

- **`InMemoryStore`** – An in-memory key-value store that persists data independently of any conversation thread. Data stored here is accessible from ANY thread as long as the Python process is running.
- **Namespace** – A tuple that organizes stored data like a folder path, e.g. `("users", "troeff", "facts")`. This scopes data to a specific user so different users don't see each other's data.
- **`ToolRuntime[AgentContext]`** – A special parameter type for tools that gives them access to shared infrastructure (the store, context, etc.) at runtime — without the AI model needing to pass it explicitly.
- **`AgentContext`** – A `TypedDict` that carries caller-supplied context into tools (e.g., `user_id`). Passed via `context=...` when invoking the agent.
- **`store.put(namespace, key, value)`** – Saves a fact to the store.
- **`store.search(namespace)`** – Retrieves all facts stored under a namespace.

## Checkpointer vs Store

| Checkpointer | Store |
|---|---|
| Remembers the conversation **within one thread** | Remembers facts **across all threads** |
| Stores message history | Stores structured facts (key-value) |
| Reset when `thread_id` changes | Persists regardless of `thread_id` |
| Short-term memory (per conversation) | Long-term memory (per user) |

In [ ]:
# Install required packages.
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import ToolRuntime, tool  # ToolRuntime: gives tools access to the store and context
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver  # Short-term: per-thread conversation memory
from langgraph.store.memory import InMemoryStore         # Long-term: cross-thread user memory
from pydantic import SecretStr
from typing import List, TypedDict  # TypedDict: defines a typed dictionary (used for AgentContext)

api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
class CustomAgentContext(TypedDict):
    user_id: str

In [ ]:
# Create BOTH a checkpointer and a store.
# They serve complementary purposes:
#   - checkpointer: remembers THIS conversation's message history (per thread)
#   - store: remembers facts about the USER across ALL conversations (cross-thread)
checkpointer = InMemorySaver()
store = InMemoryStore()

In [ ]:
# --- Tool: Save a user fact to the long-term store ---
@tool
def remember_user_facts(key: str, value: str, runtime: ToolRuntime[CustomAgentContext]) -> str:
    """
    Extract durable user facts from a user message and store them in long-term memory. Example: "key: allergy; value: The user is allergic to nuts.", "key: hobbies; value: The user can play the guitar."

    Args:
        key: A unique identifier of the fact.
        value: The fact itself.
    """
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    prev_item = runtime.store.get(namespace, "auto_extracted_facts")

    facts_dict = prev_item.value if prev_item is not None else {}
    facts_dict[key] = value

    runtime.store.put(namespace, "auto_extracted_facts", facts_dict)
    return "OK"


# --- Tool: Retrieve all stored facts for the current user ---
@tool
def recall_user_facts(runtime: ToolRuntime[CustomAgentContext]) -> str:
    """
    Recall previously stored long-term facts about the user.
    """
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    result = runtime.store.search(namespace, limit=20)
    if not result:
        return "No facts stored."

    return '\n===\n'.join(f"{facts_group.key}:\n{'\n'.join(f'- {key}: \"{value}\"' for key, value in facts_group.value.items())}" for facts_group in result)

In [ ]:
# Build the agent with both the checkpointer (short-term) and the store (long-term).
# The system prompt explicitly instructs the agent HOW to use the memory tools —
# this is important because by default the model won't know when to call them.
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key, reasoning_effort="low"),
    tools=[remember_user_facts, recall_user_facts],
    system_prompt=f"""
                  You are a polite and helpful personal assistant. You should use frequently the \"{remember_user_facts.name}\" tool to store information about the user that can be useful in future.
                  At the start of each iteration, ALWAYS use the \"{recall_user_facts.name}\" tool.
                  Be friendly - include known facts in the conversation to make the user feel special.
                  Proactively store useful data about the users - name, hobbies, plans, needs, etc.
                  """,
    checkpointer=checkpointer,
    store=store,
    context_schema=CustomAgentContext
)

interact = agent | RunnableLambda(lambda res: print_conversation(res["messages"]))

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("Hello! My name is Tony and I would like you to help me with the management of my personal notes and timeline.")]
    },
    config={
        "configurable": {
            "thread_id": "troeff_1"
        }
    },
    context={
        "user_id": "troeff"  # Passed to tools via ToolRuntime — scopes store access to this user
    }
)

In [ ]:
# NOTE: We are passing a different `thread_id` - this is a different conversation (a few days after)
# The expected result - the agent should know (at least) my name.
interact.invoke(
    input={
        "messages": [HumanMessage("I want to plan a business meeting for today, 15:00.")]
    },
    config={
        "configurable": {
            "thread_id": "troeff_2"
        }
    },
    context={
        "user_id": "troeff"  # Same user_id — agent will find Tony's facts in the store
    }
)

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("Hey! It's George. I need you to help me with the planning of a trip. I don't have much time left to waste.")]
    },
    config={
        "configurable": {
            "thread_id": "michael_3"
        }
    },
    context={
        "user_id": "michael"
    }
)